# optimal_trader trading_app Contract Replay

This notebook is the contract-first replacement for the old `optimal_trader/notebooks/trading_app.ipynb` option-backtest notebook. It does not import `optimal_trader`, does not run Streamlit, and does not include live trading logic.

The strategy path is:

`saved optimal_trader artifacts -> quant-warehouse feature panel -> scored_panel -> action_tape -> trade_windows -> optional options replay`


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "quant_orchestrator").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PROJECT_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_orchestrator.platforms.backtesting_frameworks.optimal_trader import (
    OptimalTraderArtifactReplayConfig,
    run_optimal_trader_artifact_replay,
)
from quant_orchestrator.platforms.backtesting_frameworks.strategy_artifacts import read_strategy_artifacts

REPO_ROOT, PROJECT_ROOT


## Configuration

Use cached panels while iterating. Turn off `RUN_FMP_SYNTHETIC_OPTIONS` when validating only the equity strategy contract.

In [ ]:
ARTIFACT_DIR = PROJECT_ROOT / "optimal_trader" / "artifacts" / "raw_stack"
OUTPUT_DIR = REPO_ROOT / "artifacts" / "trading_app_contract_replay"

FEATURE_START = "2020-01-01"
BACKTEST_START = "2021-01-01"
END_DATE = "2026-06-23"
TOP_K = 20
COMPONENT_THRESHOLD = 0.50
PRICE_PROVIDER = "fmp"
MAX_SYMBOLS = 0
REUSE_FEATURE_PANEL = True
REUSE_SCORED_PANEL = True
RUN_FMP_SYNTHETIC_OPTIONS = False
OPTION_WORKERS = 1

config = OptimalTraderArtifactReplayConfig(
    artifact_dir=ARTIFACT_DIR,
    output_dir=OUTPUT_DIR,
    feature_start=FEATURE_START,
    backtest_start=BACKTEST_START,
    end_date=END_DATE,
    top_k=TOP_K,
    component_threshold=COMPONENT_THRESHOLD,
    price_provider=PRICE_PROVIDER,
    max_symbols=MAX_SYMBOLS,
    reuse_feature_panel=REUSE_FEATURE_PANEL,
    reuse_scored_panel=REUSE_SCORED_PANEL,
    run_fmp_synthetic_options=RUN_FMP_SYNTHETIC_OPTIONS,
    option_workers=OPTION_WORKERS,
)
config


In [ ]:
result = run_optimal_trader_artifact_replay(config)
bundle = read_strategy_artifacts(OUTPUT_DIR)

assert bundle.scored_panel is not None and not bundle.scored_panel.empty
assert bundle.action_tape is not None
assert bundle.trade_windows is not None
assert bundle.strategy_name == "optimal_trader.trading_app"

result.summary


In [ ]:
display(pd.DataFrame([result.summary["performance"]]))
display(result.rule_replay.action_tape.head(20))
display(result.rule_replay.trade_windows.head(20))


In [ ]:
manifest_path = OUTPUT_DIR / "strategy_artifacts_manifest.json"
standard_files = sorted(path.name for path in OUTPUT_DIR.glob("*.parquet"))
{
    "manifest": str(manifest_path),
    "strategy_name": bundle.strategy_name,
    "scored_rows": 0 if bundle.scored_panel is None else len(bundle.scored_panel),
    "action_rows": 0 if bundle.action_tape is None else len(bundle.action_tape),
    "trade_windows": 0 if bundle.trade_windows is None else len(bundle.trade_windows),
    "parquet_files": standard_files,
}
